### 📥 Instalações
#### 🐍 Python: 3.12.5

In [ ]:
pip install torch numpy pandas matplotlib carbontracker eco2ai setuptools==68.2.2

In [ ]:
pip install git+https://github.com/google-research/timesfm git+https://github.com/llm4time/llm4series

### 📦 Importações

In [90]:
import numpy as np
import pandas as pd
import llm4series as ls
import matplotlib.pyplot as plt
import torch
import time

import warnings
warnings.simplefilter("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# === CarbonTracker ===
import io
import re
from contextlib import redirect_stdout
from carbontracker.tracker import CarbonTracker as Tracker

# === Eco2AI ===
import eco2ai
eco2ai_tracker = eco2ai.Tracker(project_name="eco2ai", file_name="eco2ai_emissions.csv", ignore_warnings=True)

### 🌱 Pegada de carbono

In [43]:
class CarbonTracker:
  def __init__(self, epochs=1, verbose=1):
    self.epochs = epochs
    self.verbose = verbose
    self.buffer = io.StringIO()
    self.tracker = None
    self.co2_g = None
    self.energy_kwh = None

  def __enter__(self):
    self._stdout_ctx = redirect_stdout(self.buffer)
    self._stdout_ctx.__enter__()
    self.tracker = Tracker(epochs=self.epochs, verbose=self.verbose)
    self.tracker.epoch_start()
    return self

  def __exit__(self, exc_type, exc, tb):
    self.tracker.epoch_end()
    self.tracker.stop()
    self._stdout_ctx.__exit__(exc_type, exc, tb)
    output = self.buffer.getvalue()

    match_co2 = re.search(r"CO2eq:\s*([\d\.]+)\s*g", output)
    self.co2_g = float(match_co2.group(1)) if match_co2 else None

    match_energy = re.search(r"Energy:\s*([\d\.]+)\s*kWh", output)
    self.energy_kwh = float(match_energy.group(1)) if match_energy else None

# 📊 1. Carregamento dos dados

In [91]:
def read_file(name, path, start, end, periods):
    ts = ls.read_file(path, index_col="date")
    X, y = ts.split(start=start, end=end, periods=periods)
    print(f"📈 {path}")
    print(f"    ✅ Total de períodos de treino: {len(X)}")
    print(f"    ✅ Total de períodos de teste: {len(y)}")
    y.name = "Real"
    return X, y, name, periods

electricity = read_file(name="Electricity", path="data/Electricity.csv", start="2013-06-28 07:00:01", end="2013-09-06 6:00:01", periods=24)
etth2 = read_file(name="ETTh2", path="data/ETTh2.csv", start="2016-07-12 06:00:00", end="2016-09-20 05:00:00", periods=24)
traffic = read_file(name="Traffic", path="data/Traffic.csv", start="2018-01-10 03:00:00", end="2018-03-21 02:00:00", periods=24)
covid19 = read_file(name="Covid-19", path="data/Covid-19.csv", start="2021-03-12 0:00:00", end="2022-03-10 0:00:00", periods=7)
retail = read_file(name="Retail", path="data/Retail.csv", start="2017-01-01 0:00:00", end="2017-12-30 0:00:00", periods=7)
wike2000 = read_file(name="Wike2000", path="data/Wike2000.csv", start="2012-07-08 0:00:00", end="2013-07-06 0:00:00", periods=7)
carbon = read_file(name="Carbon", path="data/Carbon.csv", start="2025-02-14 0:00:00", end="2026-02-12 00:00:00", periods=7)

📈 data/Electricity.csv
    ✅ Total de períodos de treino: 1680
    ✅ Total de períodos de teste: 24
📈 data/ETTh2.csv
    ✅ Total de períodos de treino: 1680
    ✅ Total de períodos de teste: 24
📈 data/Traffic.csv
    ✅ Total de períodos de treino: 1680
    ✅ Total de períodos de teste: 24
📈 data/Covid-19.csv
    ✅ Total de períodos de treino: 364
    ✅ Total de períodos de teste: 7
📈 data/Retail.csv
    ✅ Total de períodos de treino: 364
    ✅ Total de períodos de teste: 7
📈 data/Wike2000.csv
    ✅ Total de períodos de treino: 364
    ✅ Total de períodos de teste: 7
📈 data/Carbon.csv
    ✅ Total de períodos de treino: 364
    ✅ Total de períodos de teste: 7


# ⚙️ 2. TimesFM

In [80]:
import timesfm
from timesfm.timesfm_2p5 import timesfm_2p5_torch
torch.set_float32_matmul_precision("high")
model = timesfm_2p5_torch.TimesFM_2p5_200M_torch.from_pretrained(
  "google/timesfm-2.5-200m-pytorch"
)

model.compile(
    timesfm.ForecastConfig(
        max_context=1024,
        max_horizon=256,
        normalize_inputs=True,
        use_continuous_quantile_head=True,
        force_flip_invariance=True,
        infer_is_positive=True,
        fix_quantile_crossing=True,
    )
)

[INFO] HTTP Request: HEAD https://huggingface.co/google/timesfm-2.5-200m-pytorch/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
[INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/timesfm-2.5-200m-pytorch/1d952420fba87f3c6dee4f240de0f1a0fbc790e3/config.json "HTTP/1.1 200 OK"
[INFO] Downloading checkpoint from Hugging Face repo google/timesfm-2.5-200m-pytorch
[INFO] HTTP Request: HEAD https://huggingface.co/google/timesfm-2.5-200m-pytorch/resolve/main/model.safetensors "HTTP/1.1 302 Found"
[INFO] Loading checkpoint from: /home/wesley/.cache/huggingface/hub/models--google--timesfm-2.5-200m-pytorch/snapshots/1d952420fba87f3c6dee4f240de0f1a0fbc790e3/model.safetensors


# 🚀 3. Execução dos experimentos

In [118]:
data = [electricity, etth2, traffic, covid19, retail, wike2000, carbon]
results = []

for X, y, name, horizon in data:
    start = time.time()
    eco2ai_tracker.start()
    with CarbonTracker() as carbontracker:
        forecast, _ = model.forecast(horizon=horizon, inputs=[X.values])
    end = time.time()

    carbontracker_g = carbontracker.co2_g
    carbontracker_kWh = carbontracker.energy_kwh
    carbontracker_Wh = carbontracker_kWh * 1000

    eco2ai_tracker.stop()
    eco2ai_data = pd.read_csv("eco2ai_emissions.csv").iloc[-1]
    eco2ai_kWh = eco2ai_data["power_consumption(kWh)"]
    eco2ai_Wh = eco2ai_kWh * 1000
    eco2ai_kg = eco2ai_data["CO2_emissions(kg)"]
    eco2ai_g = eco2ai_kg * 1000

    y = y.values.tolist()
    forecast = forecast[0].tolist()
    results.append({
        "Dataset": name,
        "Model": "TimesFM-2.5-200M",
        "Parameters": "max_context=1024, max_horizon=256, normalize_inputs=True, use_continuous_quantile_head=True, force_flip_invariance=True, infer_is_positive=True, fix_quantile_crossing=True",
        "SMAPE": ls.smape(y, forecast),
        "MAE": ls.mae(y, forecast),
        "RMSE": ls.rmse(y, forecast),
        "CarbonTracker CO₂ (g)": carbontracker_g,
        "CarbonTracker Energy Consumption (Wh)": carbontracker_Wh,
        "Eco2AI CO₂ (g)": eco2ai_g,
        "Eco2AI Energy Consumption (Wh)": eco2ai_Wh,
        "Time (s)": end - start,
        "Actual Values": y,
        "Predicted Values": forecast
    })

[INFO] Adding job tentatively -- it will be properly scheduled when the scheduler starts
[INFO] Added job "Tracker._func_for_sched" to job store "default"
[INFO] Scheduler started
[INFO] Requested http://ipinfo.io/json
[INFO] Removed job job
[INFO] Scheduler has been shut down
[INFO] Adding job tentatively -- it will be properly scheduled when the scheduler starts
[INFO] Added job "Tracker._func_for_sched" to job store "default"
[INFO] Scheduler started
[INFO] Requested http://ipinfo.io/json
[INFO] Removed job job
[INFO] Scheduler has been shut down
[INFO] Adding job tentatively -- it will be properly scheduled when the scheduler starts
[INFO] Added job "Tracker._func_for_sched" to job store "default"
[INFO] Scheduler started
[INFO] Requested http://ipinfo.io/json
[INFO] Removed job job
[INFO] Scheduler has been shut down
[INFO] Adding job tentatively -- it will be properly scheduled when the scheduler starts
[INFO] Added job "Tracker._func_for_sched" to job store "default"
[INFO] Sche

# 💾 4. Salvar os resultados

In [119]:
import os
os.makedirs("experiments", exist_ok=True)
pd.DataFrame(results).to_csv("experiments/TimesFM-2.5-200M.csv", index=False)

# 📈 5. Gráfico Real x Predito

In [120]:
idx = 0
real = ls.UniTimeSeries(results[idx]["Actual Values"], name="Real")
prediction = ls.UniTimeSeries(results[idx]["Predicted Values"], name="Predicted")
ls.plot(real, prediction, title=results[idx]["Dataset"], kind="line")